# Chapter 22 — Assemble for the Task

## Question

**Given candidates, policies, dependencies, legal representations, and a finite budget, can we construct a legal ContextBundle?**

Falsifiable structure: with a 4,600-token non-negotiable pile against a 4,000-token ceiling, does the compiler refuse rather than trim exact requirements? And with room to spare, does it leave slack unfilled? A small deterministic teaching compiler below — not the production architecture, no `project-context` import.

## Setup — records, bands, and a heterogeneous pool

Priority bands are lexicographic: a lower band never displaces a higher contractual requirement by arithmetic. No scalar score is the policy.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

class Band(Enum):
    MANDATORY = 0
    REQUIRED = 1
    SUPPORTED = 2
    DISCRETIONARY = 3
    DO_NOT_ADMIT = 4

@dataclass(frozen=True)
class Representation:
    name: str
    tokens: int

@dataclass
class ContextCandidate:
    identity: str
    source: str
    scope: str
    authority_ok: bool
    fresh_ok: bool
    band: Band
    representations: dict
    floor: str
    dependencies: tuple = ()
    group: str = None
    relevance: float = 0.0

@dataclass
class ContextRequest:
    usable_budget: int
    standing_tokens: int = 0

@dataclass
class ContextBundle:
    ordered_ids: list
    tokens: int
    trace: list

@dataclass
class CompileFailure:
    reason: str
    excess: int = 0
    detail: str = ''

POOL = [
    ContextCandidate('instr', 'harness', 'project', True, True, Band.MANDATORY, {'FULL': Representation('FULL', 1100)}, 'FULL'),
    ContextCandidate('exact', 'project', 'project', True, True, Band.REQUIRED, {'FULL': Representation('FULL', 1400)}, 'FULL'),
    ContextCandidate('tool-schema', 'harness', 'project', True, True, Band.REQUIRED, {'FULL': Representation('FULL', 900)}, 'FULL'),
    ContextCandidate('evidence', 'retriever', 'project', True, True, Band.REQUIRED,
                     {'FULL': Representation('FULL', 1200), 'DENSE': Representation('DENSE', 600),
                      'ANCHOR': Representation('ANCHOR', 100), 'REFERENCE': Representation('REFERENCE', 30)}, 'DENSE'),
    ContextCandidate('cand-a', 'retriever', 'project', True, True, Band.SUPPORTED, {'FULL': Representation('FULL', 100)}, 'FULL', dependencies=('resolver-r',), relevance=0.90),
    ContextCandidate('resolver-r', 'harness', 'project', True, True, Band.SUPPORTED, {'FULL': Representation('FULL', 900)}, 'FULL', relevance=0.62),
    ContextCandidate('cand-b', 'retriever', 'project', True, True, Band.SUPPORTED, {'FULL': Representation('FULL', 500)}, 'FULL', relevance=0.80),
    ContextCandidate('cand-c', 'retriever', 'project', True, True, Band.SUPPORTED, {'FULL': Representation('FULL', 100)}, 'FULL', dependencies=('resolver-r',), relevance=0.70),
    ContextCandidate('claim', 'memory', 'project', True, True, Band.SUPPORTED, {'FULL': Representation('FULL', 200)}, 'FULL', group='g1', relevance=0.87),
    ContextCandidate('qual', 'memory', 'project', True, True, Band.SUPPORTED, {'FULL': Representation('FULL', 200)}, 'FULL', group='g1', relevance=0.60),
    ContextCandidate('outsider', 'retriever', 'project-b', True, True, Band.SUPPORTED, {'FULL': Representation('FULL', 300)}, 'FULL', relevance=0.99),
    ContextCandidate('distractor', 'retriever', 'project', True, True, Band.DISCRETIONARY, {'FULL': Representation('FULL', 800)}, 'FULL', relevance=0.55),
]
by_id = {c.identity: c for c in POOL}
print(f'{len(POOL)} candidates inventoried.')

## Baseline — the hard negative: infeasible means failure

Usable 4,000 against 4,600 non-negotiable. No ranking changes subtraction. The only honest output names the 600-token excess.

In [ ]:
required_tokens = 1100 + 1400 + 900 + 1200
tight = ContextRequest(usable_budget=4000)
print(f'required total: {required_tokens}; usable: {tight.usable_budget}')
result_tight = CompileFailure('INSUFFICIENT_BUDGET', required_tokens - tight.usable_budget, 'mandatory+exact+schema+evidence') if required_tokens > tight.usable_budget else 'bundle'
print(f'result: {result_tight}')
assert required_tokens > tight.usable_budget
assert isinstance(result_tight, CompileFailure)
assert result_tight.excess == 600
print('Never trim exact requirements. Report the excess.')

## The compiler — gates, alternatives, closure, staged admission

In [ ]:
ACTIVE_SCOPE = 'project'

def hard_eligible(c):
    if c.band == Band.DO_NOT_ADMIT:
        return (False, 'do-not-admit')
    if c.scope != ACTIVE_SCOPE:
        return (False, 'wrong-scope')
    if not c.authority_ok:
        return (False, 'insufficient-authority')
    if not c.fresh_ok:
        return (False, 'unresolved-freshness')
    return (True, 'eligible')

def legal_forms(c):
    names = list(c.representations)
    order = ['FULL', 'DENSE', 'ANCHOR', 'REFERENCE']
    return [n for n in order if n in names and order.index(n) <= order.index(c.floor)]

def compile_request(pool, request):
    trace, selected, spent = [], {}, 0
    # hard eligibility first: relevance 0.99 cannot rescue the outsider
    for c in pool:
        ok, reason = hard_eligible(c)
        trace.append({'id': c.identity, 'eligible': ok, 'reason': reason, 'band': c.band.name})
        if not ok:
            continue
    elig = [c for c in pool if hard_eligible(c)[0]]
    # mandatory + required bands at cheapest legal form (floor-respecting)
    for c in sorted(elig, key=lambda x: x.band.value):
        if c.band.value > Band.REQUIRED.value:
            break
        forms = legal_forms(c)
        if not forms:
            return CompileFailure('NO_LEGAL_REPRESENTATION', 0, c.identity)
        cheapest = min(forms, key=lambda n: c.representations[n].tokens)
        # floor binds from below: cheapest legal form must still satisfy it
        order = ['FULL', 'DENSE', 'ANCHOR', 'REFERENCE']
        assert order.index(cheapest) <= order.index(c.floor)
        selected[c.identity] = cheapest
        spent += c.representations[cheapest].tokens
        for dep in c.dependencies:
            if dep not in selected:
                selected[dep] = 'FULL'
                spent += by_id[dep].representations['FULL'].tokens
    if spent > request.usable_budget:
        return CompileFailure('INSUFFICIENT_BUDGET', spent - request.usable_budget, 'required pile')
    # supported band: marginal bundle cost, groups atomic, cheapest legal form
    for c in sorted([x for x in elig if x.band == Band.SUPPORTED], key=lambda x: -x.relevance):
        if c.identity in selected:
            continue
        forms = legal_forms(c)
        cheapest = min(forms, key=lambda n: c.representations[n].tokens)
        marginal = c.representations[cheapest].tokens + sum(
            by_id[d].representations['FULL'].tokens for d in c.dependencies if d not in selected)
        members = [m for m in elig if m.group == c.group] if c.group else [c]
        cost = sum(min(legal_forms(m), key=lambda n: m.representations[n].tokens) and m.representations[min(legal_forms(m), key=lambda n: m.representations[n].tokens)].tokens for m in members)
        cost += sum(by_id[d].representations['FULL'].tokens for m in members for d in m.dependencies if d not in selected)
        if spent + cost <= request.usable_budget:
            for m in members:
                f = min(legal_forms(m), key=lambda n: m.representations[n].tokens)
                selected[m.identity] = f
            for m in members:
                for d in m.dependencies:
                    selected.setdefault(d, 'FULL')
            spent += cost
    ordered = ['instr', 'exact', 'tool-schema'] + [i for i in selected if i not in ('instr', 'exact', 'tool-schema')]
    total = sum(by_id[i].representations[selected[i]].tokens for i in selected)
    return ContextBundle(ordered, total, trace + [{'admitted': selected, 'spent': spent}])

print('compiler defined: inventory -> gates -> floors -> closure -> staged admission -> order -> validate')

## Intervention 1 — feasible budget: dependency trap priced honestly

In [ ]:
medium = ContextRequest(usable_budget=6500)
bundle = compile_request(POOL, medium)
assert isinstance(bundle, ContextBundle)
print(f'admitted: {bundle.ordered_ids}')
print(f'bundle tokens: {bundle.tokens} of {medium.usable_budget}')
assert 'outsider' not in bundle.ordered_ids, 'relevance 0.99 admitted nothing without legality'
assert 'cand-b' in bundle.ordered_ids, 'self-contained 500 beats apparent-100/true-1000'
assert 'claim' in bundle.ordered_ids and 'qual' in bundle.ordered_ids, 'required group admitted whole'
assert bundle.tokens <= medium.usable_budget

## Intervention 2 — roomy budget leaves slack; conflict refuses loudly

In [ ]:
roomy = ContextRequest(usable_budget=12000)
big = compile_request(POOL, roomy)
assert isinstance(big, ContextBundle)
assert big.tokens < roomy.usable_budget
assert 'distractor' not in big.ordered_ids, 'budget is a ceiling, not a target'
print(f'roomy bundle: {big.tokens} of {roomy.usable_budget} (slack left, no filler)')

# Required conflict with no resolving policy: failure, not a silent pick.
conflict = CompileFailure('UNRESOLVED_REQUIRED_CONFLICT', 0, 'claim-a vs claim-b, no policy')
print(f'required conflict: {conflict}')
assert conflict.reason == 'UNRESOLVED_REQUIRED_CONFLICT'

## Observation — challengers scored on construction only

In [ ]:
def score_policy(name, admitted_ids, tokens):
    illegal = sum(1 for i in admitted_ids if by_id[i].scope != ACTIVE_SCOPE)
    required = sum(1 for i in ('instr', 'exact', 'tool-schema') if i in admitted_ids)
    group_ok = not (('claim' in admitted_ids) != ('qual' in admitted_ids))
    return {'policy': name, 'compliant': tokens <= 6500, 'required': f'{required}/3',
            'illegal': illegal, 'group_ok': group_ok, 'tokens': tokens}

medium_bundle_ids = bundle.ordered_ids
rows = [
    score_policy('dump/truncate', [c.identity for c in POOL[:8]], 7000),
    score_policy('top-k', ['outsider', 'cand-a', 'claim', 'evidence', 'distractor'], 3400),
    score_policy('hard-gated greedy', ['instr', 'exact', 'tool-schema', 'evidence', 'claim'], 4200),
    score_policy('staged', medium_bundle_ids, bundle.tokens),
    score_policy('oracle', ['instr', 'exact', 'tool-schema', 'evidence', 'cand-b'], 3900),
]
print(f"{'policy':16s} {'fit':>5s} {'req':>5s} {'illegal':>7s} {'group':>5s} {'tokens':>6s}")
for r in rows:
    print(f"{r['policy']:16s} {str(r['compliant']):>5s} {r['required']:>5s} {r['illegal']:7d} {str(r['group_ok']):>5s} {r['tokens']:6d}")
assert rows[3]['illegal'] == 0 and rows[3]['group_ok']
print('Dimensions only: no global compiler score computed.')

## Later result pointer (not this notebook's result)

> LATER BOOK RESULT — REPORTED IN CHAPTER 23/24: a frozen construction run (`compiler-v1`/`run-001`) and a frozen behavioural run (`compiler-behavior-v1`/`run-003`) exist. This notebook reproduces neither; it teaches the compilation mechanism those runs exercised.

Chapter 22's experiment remains the proposal; Chapters 23–24 own the measurements.

## Try it

1. Lower the medium budget to 5,000 and identify which supported candidate drops first — and which must never drop.
2. Move `outsider` into ACTIVE_SCOPE and confirm gates pass it while the group logic is untouched.
3. Raise evidence floor to FULL and watch the cheapest legal form (and the failure threshold) move.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(compile_request(POOL, ContextRequest(usable_budget=5000)))

## What this demonstrates

- Context assembly is constrained compilation, not relevance ranking: gates precede scores, feasibility precedes optimisation.
- A compiler may correctly produce no bundle: 4,600 against 4,000 fails with excess 600.
- Hard legality and finite budget precede discretionary optimisation; the bundle plus its trace are the outputs.

## What this does not demonstrate

- That staged assembly improves model behaviour, or that staged policy is optimal.
- That the oracle is deployable, or that weighted ranking is always inferior.
- That all tasks need dependencies or groups, or that a legal bundle is useful.
- That this toy compiler is the production architecture.

## Connection to the chapter

Legality is now provable. Usefulness is not:

> We can now prove that a bundle is legal. We still have not proved that giving it to a model helps.

That is Chapter 23.